In [1]:
import os
import sys
sys.path.append("..")

import pandas as pd
import pyKVFinder

from cavidac.constants import get_data_path, get_project_path

# Calculate Cavity Volume

In [2]:
df_param = pd.read_csv("param_kvfinder.csv")
df_param

,Molecule,Step,Probe_In,Probe_Out,Removal_Distance,Volume_Cutoff
0,1,0.25,0.4,5.0,0.2,5
1,2,0.25,0.5,6.0,0.2,80
2,3,0.25,0.5,4.0,0.2,40
3,4,0.25,0.3,7.0,0.2,5
4,5,0.25,0.5,7.0,0.2,5
5,6,0.25,0.3,7.5,0.2,75
6,7,0.25,0.4,8.0,0.2,45
7,8,0.25,0.4,7.0,0.2,5
8,9,0.25,0.4,8.0,0.2,5
9,10,0.25,0.4,6.0,0.2,20


In [9]:
volumes = []

for mol in range(14):

    host = os.path.join(get_data_path(), 'molecules', 'pdb_calix', f'{mol+1}.pdb')

    step = df_param.iloc[mol].Step.item()
    probe_in = df_param.iloc[mol].Probe_In.item()
    probe_out = df_param.iloc[mol].Probe_Out.item()

    removal_distance = df_param.iloc[mol].Removal_Distance.item()
    volume_cutoff = df_param.iloc[mol].Volume_Cutoff.item()

    # Atomic information
    atomic = pyKVFinder.read_pdb(host)

    # Grid dimensions
    vertices = pyKVFinder.get_vertices(atomic, probe_out=probe_out, step=step)

    # Cavity detection
    ncav, cavities = pyKVFinder.detect(
        atomic,
        vertices,
        step=step,
        probe_out=probe_out,
        probe_in=probe_in,
        removal_distance=removal_distance,
        volume_cutoff=volume_cutoff
    )

    # Spatial characterization
    surface, cavity_volume, area = pyKVFinder.spatial(cavities, step=step)
    
    print(f"- - - - - - - - - - - {mol+1} - - - - - - - - - - -")
    print(f"Cavity volume: {cavity_volume['KAA']} Å³")
    volumes.append(cavity_volume['KAA'])
        
        
df_param["Calculated_Volume"] = volumes

df_param.to_csv(os.path.join(str(get_project_path()), 'results', 'pyKVFinder', 'calc_volumes_kvfinder.csv'), index=False)

- - - - - - - - - - - 1 - - - - - - - - - - -
Cavity volume: 127.42 Å³
- - - - - - - - - - - 2 - - - - - - - - - - -
Cavity volume: 95.95 Å³
- - - - - - - - - - - 3 - - - - - - - - - - -
Cavity volume: 121.25 Å³
- - - - - - - - - - - 4 - - - - - - - - - - -
Cavity volume: 43.28 Å³
- - - - - - - - - - - 5 - - - - - - - - - - -
Cavity volume: 136.94 Å³
- - - - - - - - - - - 6 - - - - - - - - - - -
Cavity volume: 96.08 Å³
- - - - - - - - - - - 7 - - - - - - - - - - -
Cavity volume: 101.56 Å³
- - - - - - - - - - - 8 - - - - - - - - - - -
Cavity volume: 122.98 Å³
- - - - - - - - - - - 9 - - - - - - - - - - -
Cavity volume: 92.38 Å³
- - - - - - - - - - - 10 - - - - - - - - - - -
Cavity volume: 80.28 Å³
- - - - - - - - - - - 11 - - - - - - - - - - -
Cavity volume: 140.05 Å³
- - - - - - - - - - - 12 - - - - - - - - - - -
Cavity volume: 67.42 Å³
- - - - - - - - - - - 13 - - - - - - - - - - -
Cavity volume: 164.88 Å³
- - - - - - - - - - - 14 - - - - - - - - - - -
Cavity volume: 41.91 Å³


# Calculate and save visual. Cavity Volume for calix 1 with removal distance 0.0, 0.2, 1.0, 1.25

In [ ]:
host = os.path.join(get_data_path(), 'molecules', 'pdb_calix', '1.pdb')

step = 0.25
probe_in = 0.4
probe_out = 5.0

volume_cutoff = 5

for removal_distance in [0.0, 0.2, 1.0, 1.25]:

    # Atomic information
    atomic = pyKVFinder.read_pdb(host)

    # Grid dimensions
    vertices = pyKVFinder.get_vertices(atomic, probe_out=probe_out, step=step)

    # Cavity detection
    ncav, cavities = pyKVFinder.detect(
        atomic,
        vertices,
        step=step,
        probe_out=probe_out,
        probe_in=probe_in,
        removal_distance=removal_distance,
        volume_cutoff=volume_cutoff
    )

    # Spatial characterization
    surface, cavity_volume, area = pyKVFinder.spatial(cavities, step=step)
    
    pyKVFinder.export(os.path.join(str(get_project_path()), 'results', 'pyKVFinder', 'ex_cav_vol_rem_dist', f'calix_1_rem_dist_{removal_distance}.pdb'), cavities, surface, vertices, step=step)